### Loading a Pre-Trained Embedding Model

In [1]:
!pip install sentence-transformers scikit-learn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# 1.Load a pre-trained sentence embedding model
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

In [3]:
# 2. Print confirmation that the model is ready
print("✅ Model loaded successfully and ready to use!")

✅ Model loaded successfully and ready to use!


### Generating Word Embeddings

In [4]:
#1. Accept a list of words from the user
#2. Generate embeddings for each word
#3. Print the embedding vector size

words = input("Enter words separated by commas: ").split(",")

embeddings = model.encode(words)

for word, emb in zip(words, embeddings):
    print(f"\nWord: {word.strip()}")
    print(f"Embedding size: {len(emb)}")


Word: hey
Embedding size: 384

Word: I
Embedding size: 384

Word: am
Embedding size: 384

Word: Riya
Embedding size: 384

Word: How
Embedding size: 384

Word: you
Embedding size: 384

Word: doin
Embedding size: 384

Word: I
Embedding size: 384

Word: hope
Embedding size: 384

Word: u
Embedding size: 384

Word: r
Embedding size: 384

Word: good
Embedding size: 384

Word: U
Embedding size: 384

Word: look
Embedding size: 384

Word: pretty
Embedding size: 384

Word: in
Embedding size: 384

Word: that
Embedding size: 384

Word: dress
Embedding size: 384

Word: I
Embedding size: 384

Word: must
Embedding size: 384

Word: say
Embedding size: 384


Each word is converted into a fixed-length vector (embedding) of size 384.

This means every word is represented by 384 numerical values capturing its meaning.

The size (384) is determined by the pre-trained model (all-MiniLM-L6-v2) and remains constant.

These numbers represent semantic features, not readable values individually.

Similar words will have similar embeddings (vectors close in space).

Note: This model is designed for sentences, so word-level embeddings may be less meaningful.

### Word Semantic Similarity

In [10]:
import numpy as np
#1. Accept two words from the user
word1 = input("Enter the first word: ")
word2 = input("Enter the second word: ")

#2. Compute cosine similarity
emb1 = model.encode([word1])[0]
emb2 = model.encode([word2])[0]

prod = np.dot(emb1, emb2)
norm1 = np.linalg.norm(emb1)   
norm2 = np.linalg.norm(emb2)

similarity = prod / (norm1 * norm2)

#3. Display similarity score
print(f"Similarity between '{word1}' and '{word2}': {similarity}")

Similarity between 'Mother' and 'Father': 0.7324041128158569


### Sentence Embedding Generation

In [11]:
#1. Accept a sentence from the user
sentence = input("Enter a sentence: ")

#2. Generate its embedding
emd = model.encode([sentence])[0]

#3. Display the embedding dimension
print(f"Embedding dimension for the sentence: {len(emd)}")

Embedding dimension for the sentence: 384


### Sentence Similarity Comparison

In [14]:
#1. Accept two sentences
sentence1 = input("Enter the first sentence: ")
print(sentence1)
sentence2 = input("Enter the second sentence: ")
print(sentence2)

#2. Compute semantic similarity
emb1 = model.encode([sentence1])[0]
emb2 = model.encode([sentence2])[0]

prod = np.dot(emb1, emb2)
norm1 = np.linalg.norm(emb1)
norm2 = np.linalg.norm(emb2)

similarity = prod / (norm1 * norm2)

#3. Interpret the result (similar / not similar)
if similarity > 0.8:
    print(f"The sentences are similar with a similarity score of {similarity}")
else:
    print(f"The sentences are not similar with a similarity score of {similarity}")

How are you?
I am good
The sentences are not similar with a similarity score of 0.5169989466667175


### Building a Semantic Document Store

In [15]:
#1. Store multiple short documents in a list
#2. Generate embeddings for each document
#3. Store document-embedding pairs
documents = [
    "Artificial Intelligence is transforming industries.",
    "Machine learning is a subset of AI.",
    "Football is a popular sport worldwide.",
    "Python is widely used for data science."
]

doc_embeddings = model.encode(documents)

# Store as pairs
document_store = list(zip(documents, doc_embeddings))

print("\n✅ Document store created with embeddings!")


✅ Document store created with embeddings!


### Semantic Search Implementation

In [18]:
from sklearn.metrics.pairwise import cosine_similarity 
#1. Accept a query sentence
query = input("Enter your query: ")

#2. Generate its embedding
query_emb = model.encode([query])[0]

#3. Compute similarity with all documents
similarities = []
for doc, emb in document_store:
    sim = cosine_similarity([query_emb], [emb])[0][0]
    similarities.append(sim)

#4. Retrieve the most relevant document
most_similar_idx = np.argmax(similarities)
most_similar_doc = document_store[most_similar_idx][0]
print(f"\nMost relevant document: {most_similar_doc}\n with similarity score: {similarities[most_similar_idx]}")


Most relevant document: Machine learning is a subset of AI.
 with similarity score: 0.45257797837257385


### Keyword Search vs Semantic Search

In [27]:
sentence = input("Enter a sentence for keyword matching: ")
print(sentence)

#1. Retrieve documents using keyword matching
keyword_matches = []
for doc in documents:
    for word in sentence.split():
        if word.lower() in doc.lower().split():
            print(f"Keyword '{word}' found in document: {doc}")
            keyword_matches.append(doc)
            break

#2. Retrieve documents using semantic similarity
sent_emb = model.encode([sentence])[0]
semantic_matches = []
for doc, emb in document_store:
    sim = cosine_similarity([sent_emb], [emb])[0][0]
    print(sim)
    if sim > 0.5:  # Threshold for semantic similarity
        semantic_matches.append((doc, sim))
    

#3. Display and compare results
print("Keyword Matches:")
for doc in keyword_matches:
    print(f" - {doc}")

print("\nSemantic Matches:")
for doc, sim in semantic_matches:
    print(f" - {doc} (similarity: {sim:.2f})")

Cricket : The favourite Sport in India
Keyword 'Sport' found in document: Football is a popular sport worldwide.
0.17155945
0.124157175
0.48708197
0.106343165
Keyword Matches:
 - Football is a popular sport worldwide.

Semantic Matches:


### Mini RAG Simulation

In [28]:
query = input("\nEnter your query for RAG: ")

query_embedding = model.encode([query])
similarities = cosine_similarity(query_embedding, doc_embeddings)[0]

top_index = similarities.argmax()
retrieved_doc = documents[top_index]

augmented_prompt = f"""
Context: {retrieved_doc}

Question: {query}

Answer:
"""

print("\n✅ Augmented Prompt (RAG Step):")
print(augmented_prompt)


✅ Augmented Prompt (RAG Step):

Context: Machine learning is a subset of AI.

Question: What is AI

Answer:



RAG step: retrieve most relevant document and combine it with query to create a context-aware prompt for the LLM

### Conceptual Reflection

Embeddings convert words and sentences into dense numerical vectors that capture meaning rather than just exact wording. This allows machines to understand relationships like similarity, analogy, and context. Unlike keyword matching, embeddings represent semantic intent, so similar meanings map closer in vector space even if words differ. Semantic search becomes critical because users rarely use exact keywords, especially in natural language queries. In GenAI systems, embeddings enable efficient retrieval of relevant context from large datasets. This is essential for systems like RAG, where accurate retrieval directly impacts response quality. Without embeddings, AI systems would be limited to shallow keyword matching instead of true understanding.